# Phase 7: Candidate Clustering với Spark ML KMeans

Notebook này dùng Spark ML để phân cụm toàn bộ candidate target từ Phase 5. Mục tiêu là tìm các nhóm đặc trưng tự nhiên của candidate, không thay thế ranking `final_score`.

Nguyên tắc chạy:
- Train KMeans trên toàn bộ `analysis/candidate_target_features`, không chỉ top 100.
- Không đưa `final_score` vào feature ML vì cột này đã là tổ hợp của các score thành phần.
- Dùng Spark ML trên HDFS, không dùng pandas hoặc local fallback để train.
- Notebook đặt trong thư mục `ML/`, nhưng output dữ liệu vẫn ghi đúng layer HDFS `analysis` và `mart`.
- Gộp kết quả GEO Phase 6 dạng support score và cluster để tạo bảng mart cuối `top_candidate_targets_enriched`.

In [1]:
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Khai báo đường dẫn HDFS và cấu hình KMeans.
HDFS_BASE = "hdfs://master11:9000"
ANALYSIS_PATH = f"{HDFS_BASE}/drugtarget/data/analysis"
MART_PATH = f"{HDFS_BASE}/drugtarget/data/mart"

CANDIDATE_FEATURES_PATH = f"{ANALYSIS_PATH}/candidate_target_features"
GEO_VALIDATION_PATH = f"{ANALYSIS_PATH}/geo_validation_result"
TOP_TARGETS_PATH = f"{MART_PATH}/top_candidate_targets"

ML_CANDIDATE_FEATURES_PATH = f"{ANALYSIS_PATH}/ml_candidate_features"
ML_K_SELECTION_PATH = f"{ANALYSIS_PATH}/ml_k_selection"
CANDIDATE_CLUSTERS_PATH = f"{ANALYSIS_PATH}/candidate_clusters"
ML_CLUSTER_SUMMARY_PATH = f"{ANALYSIS_PATH}/ml_cluster_summary"
TOP_TARGETS_ENRICHED_PATH = f"{MART_PATH}/top_candidate_targets_enriched"

K_MIN = 2
K_MAX = 6
SEED = 42
EXPECTED_CANDIDATE_COUNT = 2579
EXPECTED_TOP_COUNT = 100

ML_FEATURE_COLUMNS = [
    "abs_log2FC",
    "log_weighted_degree",
    "avg_combined_score",
    "log_num_interactions",
]

# Tạo SparkSession cho Spark ML clustering.
spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase7-candidate-clustering")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Input candidate features: {CANDIDATE_FEATURES_PATH}")
print(f"Input GEO validation: {GEO_VALIDATION_PATH}")
print(f"Input top targets: {TOP_TARGETS_PATH}")
print(f"Output candidate clusters: {CANDIDATE_CLUSTERS_PATH}")
print(f"Output enriched mart: {TOP_TARGETS_ENRICHED_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 08:22:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Input candidate features: hdfs://master11:9000/drugtarget/data/analysis/candidate_target_features
Input GEO validation: hdfs://master11:9000/drugtarget/data/analysis/geo_validation_result
Input top targets: hdfs://master11:9000/drugtarget/data/mart/top_candidate_targets
Output candidate clusters: hdfs://master11:9000/drugtarget/data/analysis/candidate_clusters
Output enriched mart: hdfs://master11:9000/drugtarget/data/mart/top_candidate_targets_enriched


In [2]:
def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {', '.join(missing_columns)}")


def assert_no_nulls(frame, columns, frame_name: str) -> None:
    """Kiểm tra các cột quan trọng không null."""
    condition = None
    for column_name in columns:
        column_condition = F.col(column_name).isNull()
        condition = column_condition if condition is None else (condition | column_condition)
    bad_count = frame.filter(condition).count() if condition is not None else 0
    if bad_count != 0:
        raise AssertionError(f"{frame_name} có {bad_count} dòng null ở các cột bắt buộc: {columns}")


def hdfs_success_exists(output_path: str) -> bool:
    """Kiểm tra marker _SUCCESS trên HDFS cho output Spark."""
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    success_path = spark._jvm.org.apache.hadoop.fs.Path(output_path + "/_SUCCESS")
    fs = success_path.getFileSystem(hadoop_conf)
    return fs.exists(success_path)


candidate_features = spark.read.parquet(CANDIDATE_FEATURES_PATH)
geo_validation = spark.read.parquet(GEO_VALIDATION_PATH)
top_targets = spark.read.parquet(TOP_TARGETS_PATH)

CANDIDATE_REQUIRED_COLUMNS = [
    "gene_name",
    "gene_id_base",
    "protein_id",
    "ensp_id",
    "protein_name",
    "log2FC",
    "weighted_degree_protein",
    "avg_combined_score",
    "num_interactions_in_deg_network",
    "final_score",
]
GEO_REQUIRED_COLUMNS = [
    "gene_name_norm",
    "geo_validation_mode",
    "geo_match_status",
    "geo_num_samples_available",
    "geo_total_samples",
    "geo_coverage_rate",
    "geo_mean_expression",
    "geo_median_expression",
    "geo_mean_percentile",
    "geo_top_quartile_rate",
    "geo_support_score",
    "geo_support_level",
]
TOP_REQUIRED_COLUMNS = [
    "rank",
    "gene_name",
    "gene_id_base",
    "protein_id",
    "ensp_id",
    "protein_name",
    "log2FC",
    "p_value",
    "deg_direction",
    "degree_protein",
    "weighted_degree_protein",
    "num_interactions_in_deg_network",
    "avg_combined_score",
    "expression_score",
    "protein_network_score",
    "string_confidence_score",
    "final_score",
]

require_columns(candidate_features, CANDIDATE_REQUIRED_COLUMNS, "candidate_target_features")
require_columns(geo_validation, GEO_REQUIRED_COLUMNS, "geo_validation_result")
require_columns(top_targets, TOP_REQUIRED_COLUMNS, "top_candidate_targets")

print("Schema candidate_target_features:")
candidate_features.printSchema()
print("Preview candidate_target_features:")
candidate_features.show(10, truncate=False)
print(f"Số dòng candidate_target_features: {candidate_features.count()}")
print(f"Số dòng top_candidate_targets: {top_targets.count()}")

Schema candidate_target_features:
root
 |-- rank: integer (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)
 |-- degree_protein: long (nullable = true)
 |-- weighted_degree_protein: double (nullable = true)
 |-- num_interactions_in_deg_network: long (nullable = true)
 |-- avg_combined_score: double (nullable = true)
 |-- expression_score: double (nullable = true)
 |-- protein_network_score: double (nullable = true)
 |-- string_confidence_score: double (nullable = true)
 |-- final_score: double (nullable = true)

Preview candidate_target_features:


+----+---------+---------------+--------------------+---------------+------------+------------------+-----------------------+-------------+--------------+-----------------------+-------------------------------+------------------+-------------------+---------------------+-----------------------+-------------------+
|rank|gene_name|gene_id_base   |protein_id          |ensp_id        |protein_name|log2FC            |p_value                |deg_direction|degree_protein|weighted_degree_protein|num_interactions_in_deg_network|avg_combined_score|expression_score   |protein_network_score|string_confidence_score|final_score        |
+----+---------+---------------+--------------------+---------------+------------+------------------+-----------------------+-------------+--------------+-----------------------+-------------------------------+------------------+-------------------+---------------------+-----------------------+-------------------+
|1   |SFTPC    |ENSG00000168484|9606.ENSP00000316152

Số dòng candidate_target_features: 2579
Số dòng top_candidate_targets: 100


In [3]:
# Chuẩn bị dữ liệu ML ở đơn vị protein_id, coalesce null và giảm ảnh hưởng protein hub bằng log1p.
ml_candidate_features = (
    candidate_features
    .select(
        "gene_name",
        "gene_id_base",
        "protein_id",
        "ensp_id",
        "protein_name",
        F.col("log2FC").cast("double").alias("log2FC"),
        F.col("weighted_degree_protein").cast("double").alias("weighted_degree_protein"),
        F.col("avg_combined_score").cast("double").alias("avg_combined_score"),
        F.col("num_interactions_in_deg_network").cast("long").alias("num_interactions_in_deg_network"),
        F.col("final_score").cast("double").alias("final_score"),
    )
    .dropDuplicates(["protein_id"])
    .withColumn("abs_log2FC", F.abs(F.coalesce(F.col("log2FC"), F.lit(0.0))))
    .withColumn("weighted_degree_protein", F.coalesce(F.col("weighted_degree_protein"), F.lit(0.0)))
    .withColumn("avg_combined_score", F.coalesce(F.col("avg_combined_score"), F.lit(0.0)))
    .withColumn("num_interactions_in_deg_network", F.coalesce(F.col("num_interactions_in_deg_network"), F.lit(0).cast("long")))
    .withColumn("log_weighted_degree", F.log1p(F.col("weighted_degree_protein")))
    .withColumn("log_num_interactions", F.log1p(F.col("num_interactions_in_deg_network")))
    .cache()
)

ml_count = ml_candidate_features.count()
duplicate_protein_count = (
    ml_candidate_features.groupBy("protein_id").count().filter(F.col("count") > 1).count()
)
if ml_count != EXPECTED_CANDIDATE_COUNT:
    raise AssertionError(f"ml_candidate_features phải có {EXPECTED_CANDIDATE_COUNT} dòng, hiện có {ml_count}.")
if duplicate_protein_count != 0:
    raise AssertionError(f"ml_candidate_features có {duplicate_protein_count} protein_id bị trùng.")

assert_no_nulls(ml_candidate_features, ["protein_id", *ML_FEATURE_COLUMNS, "final_score"], "ml_candidate_features")

nan_condition = None
for column_name in ML_FEATURE_COLUMNS:
    column_condition = F.isnan(F.col(column_name))
    nan_condition = column_condition if nan_condition is None else (nan_condition | column_condition)
nan_count = ml_candidate_features.filter(nan_condition).count()
if nan_count != 0:
    raise AssertionError(f"Có {nan_count} dòng feature ML bị NaN.")

print("Schema ml_candidate_features:")
ml_candidate_features.printSchema()
print("Thống kê feature ML:")
ml_candidate_features.select(*ML_FEATURE_COLUMNS, "final_score").summary().show(truncate=False)

Schema ml_candidate_features:
root
 |-- gene_name: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- weighted_degree_protein: double (nullable = false)
 |-- avg_combined_score: double (nullable = false)
 |-- num_interactions_in_deg_network: long (nullable = false)
 |-- final_score: double (nullable = true)
 |-- abs_log2FC: double (nullable = false)
 |-- log_weighted_degree: double (nullable = true)
 |-- log_num_interactions: double (nullable = true)

Thống kê feature ML:


26/06/03 08:22:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-------------------+------------------+--------------------+---------------------+
|summary|abs_log2FC        |log_weighted_degree|avg_combined_score|log_num_interactions|final_score          |
+-------+------------------+-------------------+------------------+--------------------+---------------------+
|count  |2579              |2579               |2579              |2579                |2579                 |
|mean   |1.662410452335479 |5.774914959981161  |561.651556109962  |4.112366194048572   |0.20224356535522703  |
|stddev |0.6783264731911153|0.769893488431949  |65.51759915916071 |1.1136930184067373  |0.05519684025079134  |
|min    |1.0000242109557371|0.863733481135479  |0.0               |0.0                 |1.0964948257292768E-4|
|25%    |1.1809804770423202|5.27177830031086   |518.0361445783133 |3.332204510175204   |0.16380820336257282  |
|50%    |1.4678116005391595|5.763591077345062  |558.4444444444445 |4.143134726391533   |0.19130508563037135  |
|

In [4]:
# Tạo vector feature và scale để các feature có thang đo tương đương trước khi chạy KMeans.
assembler = VectorAssembler(inputCols=ML_FEATURE_COLUMNS, outputCol="features_raw")
ml_vector = assembler.transform(ml_candidate_features)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)
scaler_model = scaler.fit(ml_vector)
ml_scaled = scaler_model.transform(ml_vector).cache()

print("Preview vector đã scale:")
ml_scaled.select("gene_name", "protein_id", "features_raw", "features").show(10, truncate=False)

Preview vector đã scale:


+---------+--------------------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------+
|gene_name|protein_id          |features_raw                                                               |features                                                                          |
+---------+--------------------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------+
|CST4     |9606.ENSP00000217423|[1.0464913586294533,5.324988508362851,572.6481481481482,4.007333185232471] |[-0.907998018724671,-0.5844009052923399,0.16784180402387217,-0.09431055693098873] |
|LAMA4    |9606.ENSP00000230538|[1.1018511960896187,6.476873896503566,614.1450381679389,4.882801922586371] |[-0.8263856393644333,0.9117611034119726,0.801211929796996,0.6917846442460421]     |
|PITX1    |9606.ENSP00000265340|[2.48213

In [5]:
# Thử nhiều giá trị k và chọn k có silhouette score cao nhất.
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster_id",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

k_results = []
for k_value in range(K_MIN, K_MAX + 1):
    model = KMeans(
        featuresCol="features",
        predictionCol="cluster_id",
        k=k_value,
        seed=SEED,
    ).fit(ml_scaled)
    predictions = model.transform(ml_scaled)
    silhouette = evaluator.evaluate(predictions)
    k_results.append((int(k_value), float(silhouette)))
    print(f"k={k_value}, silhouette={silhouette}")

k_selection = spark.createDataFrame(k_results, ["k", "silhouette_score"]).cache()

print("K selection:")
k_selection.orderBy(F.desc("silhouette_score"), F.asc("k")).show(truncate=False)

best_k = int(k_selection.orderBy(F.desc("silhouette_score"), F.asc("k")).first()["k"])
print(f"Best k = {best_k}")

if k_selection.count() != (K_MAX - K_MIN + 1):
    raise AssertionError("ml_k_selection không có đủ số dòng cho k=2..6.")

26/06/03 08:22:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/03 08:22:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


k=2, silhouette=0.495447590208862


k=3, silhouette=0.5048944122268278


k=4, silhouette=0.396131544800113


k=5, silhouette=0.3986930383078672


k=6, silhouette=0.34725488497701457


K selection:


+---+-------------------+
|k  |silhouette_score   |
+---+-------------------+
|3  |0.5048944122268278 |
|2  |0.495447590208862  |
|5  |0.3986930383078672 |
|4  |0.396131544800113  |
|6  |0.34725488497701457|
+---+-------------------+

Best k = 3


In [6]:
# Train model cuối với best_k và lấy prediction theo candidate.
kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster_id",
    k=best_k,
    seed=SEED,
)
kmeans_model = kmeans.fit(ml_scaled)

candidate_clusters_raw = (
    kmeans_model.transform(ml_scaled)
    .select(
        "gene_name",
        "gene_id_base",
        "protein_id",
        "ensp_id",
        "protein_name",
        "log2FC",
        "abs_log2FC",
        "weighted_degree_protein",
        "log_weighted_degree",
        "avg_combined_score",
        "num_interactions_in_deg_network",
        "log_num_interactions",
        "final_score",
        F.col("cluster_id").cast("int").alias("cluster_id"),
    )
    .cache()
)

candidate_cluster_count = candidate_clusters_raw.count()
if candidate_cluster_count != EXPECTED_CANDIDATE_COUNT:
    raise AssertionError(f"candidate_clusters_raw phải có {EXPECTED_CANDIDATE_COUNT} dòng, hiện có {candidate_cluster_count}.")
assert_no_nulls(candidate_clusters_raw, ["protein_id", "cluster_id"], "candidate_clusters_raw")

print("Preview candidate_clusters_raw:")
candidate_clusters_raw.orderBy("cluster_id", F.desc("final_score")).show(20, truncate=False)

Preview candidate_clusters_raw:


+---------+---------------+--------------------+---------------+------------+-------------------+------------------+-----------------------+-------------------+------------------+-------------------------------+--------------------+-------------------+----------+
|gene_name|gene_id_base   |protein_id          |ensp_id        |protein_name|log2FC             |abs_log2FC        |weighted_degree_protein|log_weighted_degree|avg_combined_score|num_interactions_in_deg_network|log_num_interactions|final_score        |cluster_id|
+---------+---------------+--------------------+---------------+------------+-------------------+------------------+-----------------------+-------------------+------------------+-------------------------------+--------------------+-------------------+----------+
|SFTPC    |ENSG00000168484|9606.ENSP00000316152|ENSP00000316152|NULL        |-8.190919857851668 |8.190919857851668 |346.65                 |5.851196226545027  |584.8279569892474 |93                           

In [7]:
# Tạo summary cluster và gán nhãn diễn giải bằng median toàn tập.
ml_cluster_summary_base = (
    candidate_clusters_raw
    .groupBy("cluster_id")
    .agg(
        F.count("*").cast("long").alias("num_candidates"),
        F.avg("abs_log2FC").alias("avg_abs_log2FC"),
        F.avg("weighted_degree_protein").alias("avg_weighted_degree"),
        F.avg("avg_combined_score").alias("avg_combined_score"),
        F.avg("num_interactions_in_deg_network").alias("avg_num_interactions"),
        F.avg("final_score").alias("avg_final_score"),
    )
)

medians = ml_candidate_features.agg(
    F.expr("percentile_approx(abs_log2FC, 0.5)").alias("median_abs_log2FC"),
    F.expr("percentile_approx(weighted_degree_protein, 0.5)").alias("median_weighted_degree"),
).first()
median_expr = float(medians["median_abs_log2FC"])
median_network = float(medians["median_weighted_degree"])
print(f"Median abs_log2FC = {median_expr}")
print(f"Median weighted_degree_protein = {median_network}")

ml_cluster_summary = (
    ml_cluster_summary_base
    .withColumn(
        "candidate_group",
        F.when(
            (F.col("avg_abs_log2FC") >= F.lit(median_expr))
            & (F.col("avg_weighted_degree") >= F.lit(median_network)),
            F.lit("High expression + high network"),
        )
        .when(
            (F.col("avg_abs_log2FC") >= F.lit(median_expr))
            & (F.col("avg_weighted_degree") < F.lit(median_network)),
            F.lit("Expression-driven"),
        )
        .when(
            (F.col("avg_abs_log2FC") < F.lit(median_expr))
            & (F.col("avg_weighted_degree") >= F.lit(median_network)),
            F.lit("Network-driven"),
        )
        .otherwise(F.lit("Lower priority")),
    )
    .orderBy(F.desc("avg_final_score"))
    .cache()
)

candidate_clusters = (
    candidate_clusters_raw
    .join(ml_cluster_summary.select("cluster_id", "candidate_group"), on="cluster_id", how="left")
    .cache()
)

summary_rows = ml_cluster_summary.count()
summary_total = ml_cluster_summary.agg(F.sum("num_candidates").alias("total")).first()["total"]
if summary_rows != best_k:
    raise AssertionError(f"ml_cluster_summary phải có {best_k} dòng, hiện có {summary_rows}.")
if summary_total != EXPECTED_CANDIDATE_COUNT:
    raise AssertionError(f"Tổng num_candidates trong summary là {summary_total}, khác {EXPECTED_CANDIDATE_COUNT}.")
assert_no_nulls(ml_cluster_summary, ["cluster_id", "candidate_group"], "ml_cluster_summary")
assert_no_nulls(candidate_clusters, ["protein_id", "cluster_id", "candidate_group"], "candidate_clusters")

print("Cluster summary:")
ml_cluster_summary.show(truncate=False)
print("Candidate clusters preview:")
candidate_clusters.orderBy(F.desc("final_score")).show(20, truncate=False)

Median abs_log2FC = 1.4678116005391595
Median weighted_degree_protein = 317.49


Cluster summary:
+----------+--------------+------------------+-------------------+------------------+--------------------+-------------------+------------------------------+
|cluster_id|num_candidates|avg_abs_log2FC    |avg_weighted_degree|avg_combined_score|avg_num_interactions|avg_final_score    |candidate_group               |
+----------+--------------+------------------+-------------------+------------------+--------------------+-------------------+------------------------------+
|0         |315           |3.0002574921499305|415.92028571428546 |568.4919810938302 |98.93968253968254   |0.2962719438536958 |High expression + high network|
|2         |1186          |1.4733770782457636|638.8713220910623  |599.0315019041534 |182.6610455311973   |0.20911224272104928|High expression + high network|
|1         |1078          |1.4794529051451757|198.99968089053797 |518.5278551991738 |27.887755102040817  |0.1672109210297327 |Expression-driven             |
+----------+--------------+--------

+----------+---------+---------------+--------------------+---------------+------------+-------------------+------------------+-----------------------+-------------------+------------------+-------------------------------+--------------------+-------------------+------------------------------+
|cluster_id|gene_name|gene_id_base   |protein_id          |ensp_id        |protein_name|log2FC             |abs_log2FC        |weighted_degree_protein|log_weighted_degree|avg_combined_score|num_interactions_in_deg_network|log_num_interactions|final_score        |candidate_group               |
+----------+---------+---------------+--------------------+---------------+------------+-------------------+------------------+-----------------------+-------------------+------------------+-------------------------------+--------------------+-------------------+------------------------------+
|0         |SFTPC    |ENSG00000168484|9606.ENSP00000316152|ENSP00000316152|NULL        |-8.190919857851668 |8.19091

In [8]:
# Ghi các output ML vào layer analysis.
ml_candidate_features.write.mode("overwrite").option("compression", "snappy").parquet(ML_CANDIDATE_FEATURES_PATH)
k_selection.write.mode("overwrite").option("compression", "snappy").parquet(ML_K_SELECTION_PATH)
candidate_clusters.write.mode("overwrite").option("compression", "snappy").parquet(CANDIDATE_CLUSTERS_PATH)
ml_cluster_summary.write.mode("overwrite").option("compression", "snappy").parquet(ML_CLUSTER_SUMMARY_PATH)

print(f"Đã ghi ml_candidate_features: {ML_CANDIDATE_FEATURES_PATH}")
print(f"Đã ghi ml_k_selection: {ML_K_SELECTION_PATH}")
print(f"Đã ghi candidate_clusters: {CANDIDATE_CLUSTERS_PATH}")
print(f"Đã ghi ml_cluster_summary: {ML_CLUSTER_SUMMARY_PATH}")

Đã ghi ml_candidate_features: hdfs://master11:9000/drugtarget/data/analysis/ml_candidate_features
Đã ghi ml_k_selection: hdfs://master11:9000/drugtarget/data/analysis/ml_k_selection
Đã ghi candidate_clusters: hdfs://master11:9000/drugtarget/data/analysis/candidate_clusters
Đã ghi ml_cluster_summary: hdfs://master11:9000/drugtarget/data/analysis/ml_cluster_summary


In [9]:
# Gộp Phase 5 ranking, Phase 6 GEO support và Phase 7 cluster thành mart cuối.
top_targets_clean = top_targets.withColumn("gene_name_norm", F.upper(F.trim(F.col("gene_name"))))

TOP_ENRICHED_COLUMNS = [
    "rank",
    "gene_name",
    "gene_id_base",
    "protein_id",
    "ensp_id",
    "protein_name",
    "log2FC",
    "p_value",
    "deg_direction",
    "degree_protein",
    "weighted_degree_protein",
    "num_interactions_in_deg_network",
    "avg_combined_score",
    "expression_score",
    "protein_network_score",
    "string_confidence_score",
    "final_score",
    "geo_validation_mode",
    "geo_match_status",
    "geo_num_samples_available",
    "geo_total_samples",
    "geo_coverage_rate",
    "geo_mean_expression",
    "geo_median_expression",
    "geo_mean_percentile",
    "geo_top_quartile_rate",
    "geo_support_score",
    "geo_support_level",
    "cluster_id",
    "candidate_group",
]

top_candidate_targets_enriched = (
    top_targets_clean
    .join(
        geo_validation.select(
            "gene_name_norm",
            "geo_validation_mode",
            "geo_match_status",
            "geo_num_samples_available",
            "geo_total_samples",
            "geo_coverage_rate",
            "geo_mean_expression",
            "geo_median_expression",
            "geo_mean_percentile",
            "geo_top_quartile_rate",
            "geo_support_score",
            "geo_support_level",
        ),
        on="gene_name_norm",
        how="left",
    )
    .join(
        candidate_clusters.select("protein_id", "cluster_id", "candidate_group"),
        on="protein_id",
        how="left",
    )
    .select(*TOP_ENRICHED_COLUMNS)
    .orderBy("rank")
    .cache()
)

enriched_count = top_candidate_targets_enriched.count()
if enriched_count != EXPECTED_TOP_COUNT:
    raise AssertionError(f"top_candidate_targets_enriched phải có {EXPECTED_TOP_COUNT} dòng, hiện có {enriched_count}.")
assert_no_nulls(
    top_candidate_targets_enriched,
    ["rank", "gene_name", "protein_id", "geo_support_level", "cluster_id", "candidate_group"],
    "top_candidate_targets_enriched",
)

bad_geo_score_count = top_candidate_targets_enriched.filter(
    F.col("geo_support_score").isNotNull()
    & ((F.col("geo_support_score") < F.lit(0.0)) | (F.col("geo_support_score") > F.lit(1.0)))
).count()
if bad_geo_score_count != 0:
    raise AssertionError(f"Có {bad_geo_score_count} dòng geo_support_score ngoài khoảng [0, 1].")

rank_stats = top_candidate_targets_enriched.agg(
    F.min("rank").alias("min_rank"),
    F.max("rank").alias("max_rank"),
    F.countDistinct("rank").alias("distinct_rank"),
    F.count("*").alias("num_rows"),
).first().asDict()
if rank_stats["min_rank"] != 1 or rank_stats["max_rank"] != EXPECTED_TOP_COUNT or rank_stats["distinct_rank"] != EXPECTED_TOP_COUNT:
    raise AssertionError(f"Rank top enriched không liên tục 1..100: {rank_stats}")

print("Preview top_candidate_targets_enriched:")
top_candidate_targets_enriched.show(30, truncate=False)

top_candidate_targets_enriched.write.mode("overwrite").option("compression", "snappy").parquet(TOP_TARGETS_ENRICHED_PATH)
print(f"Đã ghi top_candidate_targets_enriched: {TOP_TARGETS_ENRICHED_PATH}")

Preview top_candidate_targets_enriched:


+----+---------+---------------+--------------------+---------------+------------+-------------------+-----------------------+-------------+--------------+-----------------------+-------------------------------+------------------+-------------------+---------------------+-----------------------+-------------------+-------------------------------+-------------------+-------------------------+-----------------+------------------+---------------------+---------------------+-------------------+---------------------+-------------------+--------------------+----------+------------------------------+
|rank|gene_name|gene_id_base   |protein_id          |ensp_id        |protein_name|log2FC             |p_value                |deg_direction|degree_protein|weighted_degree_protein|num_interactions_in_deg_network|avg_combined_score|expression_score   |protein_network_score|string_confidence_score|final_score        |geo_validation_mode            |geo_match_status   |geo_num_samples_available|geo_t

Đã ghi top_candidate_targets_enriched: hdfs://master11:9000/drugtarget/data/mart/top_candidate_targets_enriched


In [10]:
# Đọc lại toàn bộ output Phase 7 để kiểm tra _SUCCESS, schema và count.
for output_path in [
    ML_CANDIDATE_FEATURES_PATH,
    ML_K_SELECTION_PATH,
    CANDIDATE_CLUSTERS_PATH,
    ML_CLUSTER_SUMMARY_PATH,
    TOP_TARGETS_ENRICHED_PATH,
]:
    if not hdfs_success_exists(output_path):
        raise AssertionError(f"Output thiếu _SUCCESS: {output_path}")

ml_candidate_output = spark.read.parquet(ML_CANDIDATE_FEATURES_PATH)
k_selection_output = spark.read.parquet(ML_K_SELECTION_PATH)
candidate_clusters_output = spark.read.parquet(CANDIDATE_CLUSTERS_PATH)
cluster_summary_output = spark.read.parquet(ML_CLUSTER_SUMMARY_PATH)
top_enriched_output = spark.read.parquet(TOP_TARGETS_ENRICHED_PATH)

if ml_candidate_output.count() != EXPECTED_CANDIDATE_COUNT:
    raise AssertionError("ml_candidate_features output sai số dòng.")
if ml_candidate_output.groupBy("protein_id").count().filter(F.col("count") > 1).count() != 0:
    raise AssertionError("ml_candidate_features output có duplicate protein_id.")
if k_selection_output.count() != (K_MAX - K_MIN + 1):
    raise AssertionError("ml_k_selection output không có đúng 5 dòng.")
if candidate_clusters_output.count() != EXPECTED_CANDIDATE_COUNT:
    raise AssertionError("candidate_clusters output sai số dòng.")
if candidate_clusters_output.filter(F.col("cluster_id").isNull()).count() != 0:
    raise AssertionError("candidate_clusters có cluster_id null.")
if cluster_summary_output.count() != best_k:
    raise AssertionError("ml_cluster_summary output sai số cụm.")
if cluster_summary_output.agg(F.sum("num_candidates").alias("total")).first()["total"] != EXPECTED_CANDIDATE_COUNT:
    raise AssertionError("ml_cluster_summary tổng num_candidates không khớp.")
if cluster_summary_output.filter(F.col("candidate_group").isNull()).count() != 0:
    raise AssertionError("ml_cluster_summary có candidate_group null.")
if top_enriched_output.count() != EXPECTED_TOP_COUNT:
    raise AssertionError("top_candidate_targets_enriched output sai số dòng.")
if top_enriched_output.columns != TOP_ENRICHED_COLUMNS:
    raise AssertionError(f"Schema top_candidate_targets_enriched không đúng: {top_enriched_output.columns}")
if top_enriched_output.filter(
    F.col("geo_support_level").isNull()
    | F.col("cluster_id").isNull()
    | F.col("candidate_group").isNull()
).count() != 0:
    raise AssertionError("top_candidate_targets_enriched thiếu GEO support hoặc cluster annotation.")
if top_enriched_output.filter(
    F.col("geo_support_score").isNotNull()
    & ((F.col("geo_support_score") < F.lit(0.0)) | (F.col("geo_support_score") > F.lit(1.0)))
).count() != 0:
    raise AssertionError("top_candidate_targets_enriched có geo_support_score ngoài khoảng [0,1].")

print("Kiểm tra output Phase 7 hoàn tất.")
print("K selection output:")
k_selection_output.orderBy(F.desc("silhouette_score"), F.asc("k")).show(truncate=False)
print("Cluster summary output:")
cluster_summary_output.orderBy(F.desc("avg_final_score")).show(truncate=False)
print("Top enriched group counts:")
top_enriched_output.groupBy("geo_support_level", "candidate_group").count().orderBy(F.desc("count")).show(truncate=False)

Kiểm tra output Phase 7 hoàn tất.
K selection output:
+---+-------------------+
|k  |silhouette_score   |
+---+-------------------+
|3  |0.5048944122268278 |
|2  |0.495447590208862  |
|5  |0.3986930383078672 |
|4  |0.396131544800113  |
|6  |0.34725488497701457|
+---+-------------------+

Cluster summary output:
+----------+--------------+------------------+-------------------+------------------+--------------------+-------------------+------------------------------+
|cluster_id|num_candidates|avg_abs_log2FC    |avg_weighted_degree|avg_combined_score|avg_num_interactions|avg_final_score    |candidate_group               |
+----------+--------------+------------------+-------------------+------------------+--------------------+-------------------+------------------------------+
|0         |315           |3.0002574921499305|415.92028571428546 |568.4919810938302 |98.93968253968254   |0.2962719438536958 |High expression + high network|
|2         |1186          |1.4733770782457636|638.87132

+--------------------+------------------------------+-----+
|geo_support_level   |candidate_group               |count|
+--------------------+------------------------------+-----+
|Moderate GEO support|High expression + high network|47   |
|Limited GEO support |High expression + high network|35   |
|Not Found           |High expression + high network|18   |
+--------------------+------------------------------+-----+



In [11]:
# Giải phóng cache sau khi hoàn tất Phase 7.
ml_candidate_features.unpersist()
ml_scaled.unpersist()
k_selection.unpersist()
candidate_clusters_raw.unpersist()
ml_cluster_summary.unpersist()
candidate_clusters.unpersist()
top_candidate_targets_enriched.unpersist()
print("Hoàn tất Phase 7: Candidate Clustering với Spark ML KMeans")

Hoàn tất Phase 7: Candidate Clustering với Spark ML KMeans
